In [1]:
import ee
import geemap
import os
import geopandas as gpd

In [ ]:
Map = geemap.Map()
Map

In [7]:
landscan_global = ee.ImageCollection('projects/sat-io/open-datasets/ORNL/LANDSCAN_GLOBAL');
popcount_intervals = '<RasterSymbolizer>' + ' <ColorMap type="intervals" extended="false" >' + \
    '<ColorMapEntry color="#CCCCCC" quantity="0" label="No Data"/>' + \
    '<ColorMapEntry color="#FFFFBE" quantity="5" label="Population Count (Estimate)"/>' + \
    '<ColorMapEntry color="#FEFF73" quantity="25" label="Population Count (Estimate)"/>' + \
    '<ColorMapEntry color="#FEFF2C" quantity="50" label="Population Count (Estimate)"/>' + \
    '<ColorMapEntry color="#FFAA27" quantity="100" label="Population Count (Estimate)"/>' + \
    '<ColorMapEntry color="#FF6625" quantity="500" label="Population Count (Estimate)"/>' + \
    '<ColorMapEntry color="#FF0023" quantity="2500" label="Population Count (Estimate)"/>' + \
    '<ColorMapEntry color="#CC001A" quantity="5000" label="Population Count (Estimate)"/>' + \
    '<ColorMapEntry color="#730009" quantity="185000" label="Population Count (Estimate)"/>' + \
    '</ColorMap>' + '</RasterSymbolizer>'

dict = {
  'names': [
    '0', '1-5', '6-25', '26-50', '51-100', '101-500', '501-2500', '2501-5000',
    '5001-185000'
  ],
  'colors': [
    '#CCCCCC', '#FFFFBE', '#FEFF73', '#FEFF2C', '#FFAA27', '#FF6625', '#FF0023',
    '#CC001A', '#730009'
  ]
};

Map.addLayer(
    landscan_global.sort('system:time_start')
        .first()
        .sldStyle(popcount_intervals),
    {}, 'Population Count Estimate 2000');
Map.addLayer(
    landscan_global.sort('system:time_start', False)
        .first()
        .sldStyle(popcount_intervals),
    {}, 'Population Count Estimate 2022');

In [9]:
url = "https://services3.arcgis.com/dxRQUfTDNtfqZ301/arcgis/rest/services/County/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
gdf_mi_test = gpd.read_file(url, rows=1)
gdf_mi_test.head()

,OBJECTID,FIPSCode,Name,FeatureID,MapLayout,FIPSNum,Label,Type,CntyCode,Peninsula,MGFVersion,Shape__Area,Shape__Length,GlobalID,geometry
0,342,001,Alcona,51931f35-ce9a-491b-b699-49036d732254,landscape,1,Alcona County,County,1,Lower,V26,1.798557e+09,172519.815622,324fedbc-244e-40ef-89e2-ca15d2fdf578,"POLYGON ((-83.31858 44.51165, -83.31859 44.511..."


In [10]:
selection_layer = geemap.gdf_to_ee(gdf_mi_test)
selection_layer

In [39]:
collection = ee.ImageCollection('projects/sat-io/open-datasets/ORNL/LANDSCAN_GLOBAL')
collection

In [45]:
# get only most recent landscan dataset
landscan_2024 = collection.filterDate('2024-01-01', '2025-01-01').first()
# create raster and select the only band 'b1'
landscan_raster = landscan_2024.select('b1')
landscan_raster

### Get sum of all pixels inside polygon
- Using one county, we are get a sum of all of the population pixels 
- This will be returned as the `sum` column 
- This is for Alcona County, which has an approx pop of *10,624*

In [47]:
# get the native resolution of the raster 
scale = landscan_raster.projection().nominalScale()
# for each polygon, look at all pixels inside and compute stats
county_stats = landscan_raster.reduceRegions(
    collection=selection_layer,
    reducer=(
        ee.Reducer.count()
        .combine(ee.Reducer.sum(), sharedInputs=True)
        .combine(ee.Reducer.mean(), sharedInputs=True)
        .combine(ee.Reducer.median(), sharedInputs=True)
        .combine(ee.Reducer.minMax(), sharedInputs=True)
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
    ),
    scale=scale,
    tileScale=4,
    crs=landscan_raster.projection(),
)
geemap.ee_to_gdf(county_stats).head()['sum']

0    10113.952941
Name: sum, dtype: float64

```python
10133 is fairly close to 10624  
I'll call that a success! 
So now we are able to not only get a mean, but also the total count inside a polygon
In this case we are using a single county boundary to test
```

In [ ]:
url = "https://services3.arcgis.com/dxRQUfTDNtfqZ301/arcgis/rest/services/County/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
gdf_mi_counties = gpd.read_file(url)
gdf_mi_counties["county_id"] = gdf_mi_counties.index
gdf_mi_counties.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   OBJECTID       83 non-null     int32   
 1   FIPSCode       83 non-null     object  
 2   Name           83 non-null     object  
 3   FeatureID      83 non-null     object  
 4   MapLayout      83 non-null     object  
 5   FIPSNum        83 non-null     int32   
 6   Label          83 non-null     object  
 7   Type           83 non-null     object  
 8   CntyCode       83 non-null     object  
 9   Peninsula      82 non-null     object  
 10  MGFVersion     83 non-null     object  
 11  Shape__Area    83 non-null     float64 
 12  Shape__Length  83 non-null     float64 
 13  GlobalID       83 non-null     object  
 14  geometry       83 non-null     geometry
 15  county_id      83 non-null     int64   
dtypes: float64(2), geometry(1), int32(2), int64(1), object(10)
memory usage: 9